In [1]:
print("Hello, Chris and Daliah!")

Hello, Chris and Daliah!


In [2]:
print("Hello, Chris and Daliah!, testing functional branch") #comment test

Hello, Chris and Daliah!, testing functional branch


# Task 2: Running an Original eLCS on the Raw SUPPORT2 dataset.
* Select an appropriate LCS variant, preferably eLCS.
* Use the original LCS code without algorithmic modification.
* Run the original LCS system on the raw or minimally processed dataset.
* Report the selected LCS parameters.
* Record the baseline performance results.
* Explain any minimal processing required to make the dataset compatible with the LCS code.

For Jono and Daaliah: pip install scikit-elcs run this in your terminal to get the eLCS, same one from LCS labs. 

In [3]:
import numpy as np
import pandas as pd

from skeLCS import eLCS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import inspect

print("eLCS loaded from:")
print(inspect.getfile(eLCS))

eLCS loaded from:
c:\Users\chris\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\skeLCS\eLCS.py


In [4]:
df_raw = pd.read_csv("support2.csv")

print(df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes.value_counts()) 
# loading the dataset and displaying its shape, first few rows, and data types of each column.
# this is teh raw dataset df_raw

df_baseline = df_raw.copy(deep=True) 
# preservation of the original dataset for baseline comparison. 
# task 2 is the baseline which will be used to compare the performance of the eLCS model later on. 

(9105, 47)


,age,death,sex,hospdead,slos,d.time,dzgroup,dzclass,num.co,edu,...,crea,sod,ph,glucose,bun,urine,adlp,adls,sfdm2,adlsc
1,62.84998,0,male,0,5,2029,Lung Cancer,Cancer,0,11.0,...,1.199951,141.0,7.459961,NaN,NaN,NaN,7.0,7.0,NaN,7.0
2,60.33899,1,female,1,4,4,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,5.500000,132.0,7.250000,NaN,NaN,NaN,NaN,1.0,<2 mo. follow-up,1.0
3,52.74698,1,female,0,17,47,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,...,2.000000,134.0,7.459961,NaN,NaN,NaN,1.0,0.0,<2 mo. follow-up,0.0
4,42.38498,1,female,0,3,133,Lung Cancer,Cancer,2,11.0,...,0.799927,139.0,NaN,NaN,NaN,NaN,0.0,0.0,no(M2 and SIP pres),0.0
5,79.88495,0,female,0,16,2029,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,...,0.799927,143.0,7.509766,NaN,NaN,NaN,NaN,2.0,no(M2 and SIP pres),2.0


float64    31
int64       8
str         8
Name: count, dtype: int64


In [5]:
target = "sfdm2" # target variable for the classification task, which is the column 'sfdm2' in the dataset.

print(df_baseline[target].value_counts(dropna=False)) # inspection of the target variable. 

# incomplete values are expected in df_raw. 

df_baseline = df_baseline.dropna(subset=[target]).copy()
print("Rows after removing missing target:", len(df_baseline))
# because of the way eLCS works, the taget variables must not contain "NaN", only numerical values so this will 
# be an example of minimal preprocessing. 
# a supervised learning algorithm like eLCS requires a complete target
#  variable to learn from the data, which is why we have dropped na. 

sfdm2
<2 mo. follow-up       3123
no(M2 and SIP pres)    3061
NaN                    1400
adl>=4 (>=5 if sur)     916
SIP>=30                 564
Coma or Intub            41
Name: count, dtype: int64
Rows after removing missing target: 7705


In [6]:
# another thing is that scikit-eLCS supports continuous and discrete attributes, mixed feature types and 
# missing predictor values, but its core estimator expects the data passed into fit() to be numeric. 

# so strings will have to be encoded into numbers. sorry jono ily <3
X_df = df_baseline.drop(columns=[target]).copy()
y_raw = df_baseline[target].copy()

categorical_cols = X_df.select_dtypes(
    include=["object", "string", "category"]
).columns

print("Categorical predictors:")
print(categorical_cols.tolist()) # identify categorical columns in the dataset.

category_mappings = {}

for col in categorical_cols:
    codes, labels = pd.factorize(X_df[col], sort=True)

    X_df[col] = codes.astype(float)

    # factorize represents missing values as -1.
    # Put these back to NaN because eLCS supports missing predictor values.
    X_df.loc[X_df[col] == -1, col] = np.nan

    category_mappings[col] = labels.tolist()

    # numbers do not have hierarchy, 3 > 2 is not true. 

Categorical predictors:
['sex', 'dzgroup', 'dzclass', 'income', 'race', 'ca', 'dnr']


In [7]:
# the target variable is also categorical, so it will be encoded as well.
# but separately. 

target_encoder = LabelEncoder()

y = target_encoder.fit_transform(y_raw)

X = X_df.to_numpy(dtype=float)

print("Target mapping:") # so we know what the encoded values mean.

for i, class_name in enumerate(target_encoder.classes_):
    print(i, "=", class_name)

#data check. 
print("X shape:", X.shape)
print("y shape:", y.shape)

print("Missing predictor values:", np.isnan(X).sum())
print("Missing target values:", pd.isna(y).sum())
# Predictor missingness doesn't matter, eLCS can handle that naturally. 
# target missingness is 0, good. eLCS requires a complete target variable to learn from the data.

Target mapping:
0 = <2 mo. follow-up
1 = Coma or Intub
2 = SIP>=30
3 = adl>=4 (>=5 if sur)
4 = no(M2 and SIP pres)
X shape: (7705, 46)
y shape: (7705,)
Missing predictor values: 36948
Missing target values: 0


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
) # test train split, 80% training, 20% testing, this matches the later split as well so might as well use the same split. 

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

# stratify=y? sfdm2 is imbalanced, so stratifying the split ensures that both the training and test sets have a similar 
# distribution of the target classes. This is important for model evaluation, as it helps to avoid bias in the performance 
# metrics due to class imbalance.

print("Training class distribution:")
print(pd.Series(y_train).value_counts(normalize=True).sort_index())

print("\nTesting class distribution:")
print(pd.Series(y_test).value_counts(normalize=True).sort_index())

Training set: (6164, 46)
Test set: (1541, 46)
Training class distribution:
0    0.405256
1    0.005354
2    0.073167
3    0.118916
4    0.397307
Name: proportion, dtype: float64

Testing class distribution:
0    0.405581
1    0.005191
2    0.073329
3    0.118754
4    0.397145
Name: proportion, dtype: float64


In [9]:
# implementation beginss. 
baseline_elcs = eLCS(
    random_state=42
)
# this is our baseline model, with default parameters everywhere, those parameters is what we will be tuning
# once we move further into the tasks. 
# below are the default values from scikit-eLCS. 
#learning_iterations = 10000
#N = 1000
#p_spec = 0.5
#nu = 5
#chi = 0.8
#mu = 0.04
#theta_GA = 25
#selection_method = tournament

params = baseline_elcs.get_params()

params_df = pd.DataFrame(
    params.items(),
    columns=["Parameter", "Value"]
)

display(params_df) # this function displays the parameters of the baseline eLCS model in a tabular format,
#making it easier to review and understand the configuration of the model before training.

# gonna save it to a csv for later reference.
# this also makes reporting easier. 
params_df.to_csv(
    "task2_original_elcs_parameters.csv",
    index=False
)

,Parameter,Value
0,N,1000
1,acc_sub,0.99
2,beta,0.2
3,chi,0.8
4,delta,0.1
5,discrete_attribute_limit,10
6,do_GA_subsumption,True
7,do_correct_set_subsumption,False
8,fitness_reduction,0.1
9,init_fit,0.01


In [17]:
baseline_elcs.fit(X_train, y_train); 
print("eLCS training complete.")

# do not remove the ; idky but the scikit-eLCS output is such that jupyter clashes with it,
# so it will train perfectly fine but just wont show you nicely. ; makes it so that the chunk runs but doesnt display. 
# we can view it later= 

# boom thats the whole thing. thats our model. 
# whats actually happening in here tho? its happening in order, 1 > 2 > 3

# 1. take current patient (instance) - dont lose track of the current patient,
# 2. construct match set [M], match set is the collection of classifiers that match the current instance based on their conditions. 
# 3. construct correct set [C], the subset of the match set that contains classifiers that correctly classify the current instance.
# 4. update rule parameters/fitness 

# 5. maybe performs subsumption, which is the process of replacing a more specific classifier with a more 
# general one if the general one has better performance.

# 6. maybe performs GA genetic algorithm, which is a search heuristic that mimics the process of natural selection to
# generate high-quality solutions for optimization and search problems.

# 7. crossover/mutation generate rules. 
# 8. delete rules if the population size exceeds the maximum allowed size.
# 9. next patient (instance), repeats. each. time. 

# important note: learning_interations = 10000, means 10,000 individual learning cycles. 

eLCS training complete.


In [18]:
print("Training completed:", baseline_elcs.hasTrained)
print("Iterations completed:", baseline_elcs.explorIter)
print("Number of rules:", len(baseline_elcs.population.popSet))

Training completed: True
Iterations completed: 10000
Number of rules: 948


In [19]:
y_pred = baseline_elcs.predict(X_test)
# what is this? this is the prediction step, 
# where the trained eLCS model is used to predict the 
# target variable for the test set instances.

# an important note for later:
# baseline_elcs.score(X_test, y_test)
# the eLCS model overrides this and returns balanced score, not ordinary accuracy. 

# whats the point?
# baseline_elcs.score(xxxxx) should not be labeled as accuracy, because it is not. it is balanced accuracy.

accuracy = accuracy_score(y_test, y_pred)

balanced_acc = balanced_accuracy_score(
    y_test,
    y_pred
)

precision_macro = precision_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

recall_macro = recall_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

f1_macro = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

# all of these are to be stored and saved. 
baseline_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1"
    ],
    "Value": [
        accuracy,
        balanced_acc,
        precision_macro,
        recall_macro,
        f1_macro
    ]
})

display(baseline_results)

baseline_results.to_csv(
    "task2_original_elcs_results.csv",
    index=False
) # this is saved to a CSV aswell since everything we do later will be compared to this, nice to have a seprate 
# file for it, ya feel?

print(
    classification_report(
        y_test,
        y_pred,
        target_names=target_encoder.classes_,
        zero_division=0
    )
)

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=target_encoder.classes_,
    columns=target_encoder.classes_
)

display(cm_df)

,Metric,Value
0,Accuracy,0.718365
1,Balanced Accuracy,0.358928
2,Macro Precision,0.343498
3,Macro Recall,0.358928
4,Macro F1,0.322452


                     precision    recall  f1-score   support

   <2 mo. follow-up       0.82      0.85      0.83       625
      Coma or Intub       0.00      0.00      0.00         8
            SIP>=30       0.00      0.00      0.00       113
adl>=4 (>=5 if sur)       0.25      0.01      0.01       183
no(M2 and SIP pres)       0.65      0.94      0.77       612

           accuracy                           0.72      1541
          macro avg       0.34      0.36      0.32      1541
       weighted avg       0.62      0.72      0.64      1541



,<2 mo. follow-up,Coma or Intub,SIP>=30,adl>=4 (>=5 if sur),no(M2 and SIP pres)
<2 mo. follow-up,530,0,0,1,94
Coma or Intub,3,0,0,0,5
SIP>=30,19,0,0,2,92
adl>=4 (>=5 if sur),62,2,0,1,118
no(M2 and SIP pres),35,0,1,0,576


In [20]:
# super handy, we can export the learned rules. Pretty sure this pops up later so I think saving the 
# initial rules could be useful.
baseline_elcs.export_final_rule_population(
    headerNames=np.array(X_df.columns),
    className="sfdm2",
    filename="task2_original_elcs_rules.csv"
)

# and 

baseline_elcs.export_iteration_tracking_data(
    "task2_original_elcs_tracking.csv"
)

# theres an issue I wanna raise tho, this minimally processed dataset which the model has been trained on access to 
# the future variables... do we want that? or should future variable cleansing count as minimal preprocessing? 
# see what yall think. 